# Azure OpenAI v1 Migration

Create PAYG Azure OpenAI capacity, establish a dated legacy API, then add v1 without changing the legacy contract. No existing Azure OpenAI account, PTU, manual keys or approval inventory is required. See [README.md](README.md) for quota, permissions and costs.

### What This Sample Does

- Deploy two PAYG models with model-specific singleton pools using the existing APIM infrastructure.
- Establish both legacy baselines before adding the v1 model-in-body contract alongside them.
- Verify both v1 routes, an unknown-model rejection and both unchanged legacy requests.
- Compare routing, latency and tokens in shared test summaries, charts and a sanitized local report.

### 🛠️ Initialize Notebook Variables

Modify **USER CONFIGURATION** only. Run cells in order. Policies are loaded with `utils.read_policy_xml()` and attached to `API` definitions, following the other samples. Smoke cells use `migration_helpers.test()` to verify HTTP status and response shape with bounded readiness retries. Finish with the shared test summary, token/latency table, per-model charts and a local HTML report.

In [ ]:
import utils
import migration_helpers
from apimtypes import API, APIOperation, APIM_SKU, HTTP_VERB, INFRASTRUCTURE, AzureOpenAIModel, Region
from azure_resources import get_infra_rg_name
from console import print_ok

# ------------------------------
#    USER CONFIGURATION
# ------------------------------
rg_location         = Region.EAST_US_2
index               = 1
apim_sku            = APIM_SKU.BASICV2
deployment          = INFRASTRUCTURE.SIMPLE_APIM
sample_name         = 'aoai-v1-migration-1'  # Change the suffix for a separate sample instance on the same gateway.
primary_model       = AzureOpenAIModel.GPT_5_MINI
secondary_model     = AzureOpenAIModel.GPT_5_NANO
model_sku           = 'GlobalStandard'  # PAYG only: GlobalStandard or Standard, subject to regional model availability.
model_capacity      = 10  # Thousands of TPM; subscription quota and model-specific conversion apply.
legacy_api_version  = '2024-10-21'
tags                = ['aoai-v1-migration', 'ai-gateway']

# ------------------------------
#    SYSTEM CONFIGURATION
# ------------------------------
sample_folder = 'aoai-v1-migration'
supported_infras = [
    INFRASTRUCTURE.SIMPLE_APIM, INFRASTRUCTURE.APPGW_APIM,
    INFRASTRUCTURE.APPGW_APIM_PE, INFRASTRUCTURE.AFD_APIM_PE,
]
utils.enable_module_autoreload('migration_helpers')
migration_helpers.validate_configuration(sample_name, model_sku, model_capacity)
model_configuration = migration_helpers.build_models(sample_name, [primary_model, secondary_model], model_capacity)
rg_name = get_infra_rg_name(deployment, index)
nb_helper = utils.NotebookHelper(
    sample_folder, rg_name, rg_location, deployment, supported_infras, index = index, apim_sku = apim_sku,
)
legacy_path = f'{sample_name}/openai'
v1_path = f'{sample_name}/openai/v1'

# Policies
pol_legacy = utils.read_policy_xml('legacy.xml', sample_name = sample_folder)
pol_v1 = utils.read_policy_xml('v1.xml', sample_name = sample_folder)

# API operations
legacy_operation = APIOperation(
    'chat-completions', 'Chat Completions', '/deployments/{deployment}/chat/completions', HTTP_VERB.POST,
    'Preserves the dated Azure OpenAI contract.',
    templateParameters = [{'name': 'deployment', 'type': 'string', 'required': True}],
)
v1_operation = APIOperation(
    'chat-completions', 'v1 Chat Completions', '/chat/completions', HTTP_VERB.POST,
    'Forwards the model-in-body contract unchanged to Azure OpenAI.',
)

# APIs
legacy_api = API(
    f'{sample_name}-legacy', 'Azure OpenAI Chat Completions', legacy_path,
    'Dated deployment-in-path chat-completions contract.',
    policyXml = pol_legacy, operations = [legacy_operation], tags = tags,
)
v1_api = API(
    f'{sample_name}-v1', 'Azure OpenAI v1 Migration - v1', v1_path,
    'v1 model-in-body contract, sharing the two legacy model pools.',
    policyXml = pol_v1, operations = [v1_operation], tags = tags,
)
bicep_parameters = {
    'sampleName': {'value': sample_name}, 'location': {'value': rg_location},
    'models': {'value': model_configuration}, 'modelSku': {'value': model_sku},
}
print_ok('Notebook initialized: sample-owned PAYG resources and two isolated API paths')

### 🚀 Stage 1: Deploy PAYG Azure OpenAI and the Legacy API

`NotebookHelper.deploy_sample()` reuses the selected infrastructure or offers the standard selection/creation workflow. Bicep creates one Azure OpenAI account, two different PAYG model deployments, two model-specific pools with one backend each, an OpenAI User role assignment for APIM managed identity, and only the legacy API. It does not duplicate monitoring.

After configuration or template changes, re-run Cell 4 before deploying so the kernel rebuilds the current parameters and policy definitions. Incremental reruns use the same resource names. After stage 2 has run, rerunning stage 1 updates legacy but does not delete the existing v1 API.

In [ ]:
assert 'sample_name' in locals() and 'bicep_parameters' in locals(), 'Re-run Cell 4 to initialize the current configuration.'
assert bicep_parameters.get('sampleName') == {'value': sample_name}, 'Configuration is stale. Re-run Cell 4 before deploying.'
assert 'models' in bicep_parameters and not {'modelName', 'modelVersion', 'modelCapacity', 'modelAlias', 'legacyApiVersion'} & bicep_parameters.keys(), 'Re-run Cell 4 before deploying.'
legacy_before = {}  # Invalidate previous smoke evidence before every stage-one deployment.
v1_success = {}
legacy_after = {}
v1_rejection = None
legacy_output = nb_helper.deploy_sample({**bicep_parameters, 'apis': {'value': [legacy_api.to_dict()]}})
legacy_context = nb_helper.get_deployment_context(legacy_output)
legacy_key = migration_helpers.subscription_key(legacy_context.apis, legacy_api.name)
print_ok('Stage 1 deployed; v1 has not been added by this stage')

### ✅ Stage 1: Establish the Legacy Baseline

Treat `/<sample>/openai` as the established API path, not a newly named `/legacy` endpoint. Call both deployment names in the URL with a dated `api-version`; there is no body `model` selector. For each model, validate one warm-up request and discard its evidence, then send an identical second request through the same client. Only the second successful request is stored as **Legacy Before** and included in the table and chart, so cold-start latency is excluded. Both requests must pass the HTTP 200, completion, exact pool and model snapshot checks. This proves routing, not load balancing.

In [ ]:
legacy_payload = {
    'messages': [{'role': 'user', 'content': 'Reply with a short greeting.'}],
    'max_completion_tokens': 1024, 'reasoning_effort': 'low', 'stream': False,
}
legacy_before = {}
legacy_request_paths = {}
v1_success = {}
legacy_after = {}
v1_rejection = None
for model_config in model_configuration:
    deployment_name = model_config['deploymentName']
    legacy_request_paths[deployment_name] = f'/{legacy_path}/deployments/{deployment_name}/chat/completions?api-version={legacy_api_version}'
    result = migration_helpers.test(
        lambda: nb_helper.create_apim_requests(legacy_context.apim_gateway_url, legacy_key),
        legacy_request_paths[deployment_name], legacy_payload,
        expected_pool = model_config['poolName'], expected_model = f'{model_config["name"]}-{model_config["version"]}',
        warmup = True,
    )
    assert result.status_code == 200 and result.has_completion
    assert result.backend_pool == model_config['poolName']
    legacy_before[deployment_name] = result
    print_ok(f'Legacy exercised {result.backend_pool}: {result.model}')

### 🚀 Stage 2: Add v1 Alongside Legacy

Only proceed after the legacy smoke test succeeds. Keep the same legacy API definition, account, model snapshot and PAYG capacity. Add the v1 API with a separate subscription key and a sample-owned path, rather than reserving the gateway-wide `/openai/v1` path.

In [ ]:
assert 'sample_name' in locals() and 'bicep_parameters' in locals(), 'Re-run Cell 4 to initialize the current configuration.'
assert bicep_parameters.get('sampleName') == {'value': sample_name}, 'Configuration is stale. Re-run Cell 4 before deploying.'
assert 'models' in bicep_parameters and not {'modelName', 'modelVersion', 'modelCapacity', 'modelAlias', 'legacyApiVersion'} & bicep_parameters.keys(), 'Re-run Cell 4 before deploying.'
assert set(legacy_before) == {item['deploymentName'] for item in model_configuration}, 'Run the legacy baseline for both pools before adding v1.'
assert all(result.has_completion for result in legacy_before.values()), 'Run the legacy baseline before adding v1.'
v1_success = {}
legacy_after = {}
v1_rejection = None
v1_output = nb_helper.deploy_sample({**bicep_parameters, 'apis': {'value': [api.to_dict() for api in [legacy_api, v1_api]]}})
v1_context = nb_helper.get_deployment_context(v1_output)
v1_key = migration_helpers.subscription_key(v1_context.apis, v1_api.name)
assert migration_helpers.subscription_key(v1_context.apis, legacy_api.name) == legacy_key
print_ok('Stage 2 deployed: legacy preserved, v1 added')

### ✅ Stage 2: Verify the v1 Contract

Use `POST /<sample>/openai/v1/chat/completions` with the actual deployment name in the JSON `model` field and no dated query parameter. APIM forwards the body unchanged to `/openai/v1/chat/completions`; Azure OpenAI validates the deployment and payload.

Call both deployment names and assert the pool header and actual model snapshot for each. The v1 policy parses JSON once and appends `-pool` to the selector, but never rewrites the body. There is no dictionary or default pool. An unknown selector attempts a nonexistent pool and returns a traced HTTP 500 gateway error from `on-error`. Backend HTTP errors for resolved pools still pass through `outbound` unchanged.

In [ ]:
v1_success = {}
legacy_after = {}
v1_rejection = None
for model_config in model_configuration:
    deployment_name = model_config['deploymentName']
    v1_payload = {**legacy_payload, 'model': deployment_name}
    result = migration_helpers.test(
        lambda: nb_helper.create_apim_requests(v1_context.apim_gateway_url, v1_key),
        f'/{v1_path}/chat/completions', v1_payload,
        expected_pool = model_config['poolName'], expected_model = f'{model_config["name"]}-{model_config["version"]}',
    )
    assert result.status_code == 200 and result.has_completion
    assert result.backend_pool == legacy_before[deployment_name].backend_pool == model_config['poolName']
    v1_success[deployment_name] = result
    print_ok(f'v1 exercised {result.backend_pool}: {result.model}')
v1_rejection = migration_helpers.test(
    lambda: nb_helper.create_apim_requests(v1_context.apim_gateway_url, v1_key),
    f'/{v1_path}/chat/completions', {**legacy_payload, 'model': 'unknown-model'},
    expected_status = 500, retry_delays = (),
)
assert v1_rejection.status_code == 500 and not v1_rejection.has_completion
print_ok('v1 completion and unknown-model rejection passed')

### ✅ Final Legacy Regression

Send the exact original path, body and subscription key after v1 is installed. Expect the same legacy contract to succeed; generated text need not be identical. This is coexistence evidence for this lab, not a production migration certification.

In [ ]:
legacy_after = {}
for model_config in model_configuration:
    deployment_name = model_config['deploymentName']
    result = migration_helpers.test(
        lambda: nb_helper.create_apim_requests(v1_context.apim_gateway_url, legacy_key),
        legacy_request_paths[deployment_name], legacy_payload,
        expected_pool = model_config['poolName'], expected_model = f'{model_config["name"]}-{model_config["version"]}',
    )
    assert result.status_code == legacy_before[deployment_name].status_code == 200
    assert result.has_completion and result.backend_pool == legacy_before[deployment_name].backend_pool
    legacy_after[deployment_name] = result
    print_ok(f'Legacy regression exercised {result.backend_pool}: {result.model}')
print_ok('Legacy regression passed for both model pools')

### 📊 [Verify] Migration Evidence and Local Report

Review all seven probes together using the same `ApimTesting`, console table, `charts.BarChart` and `HtmlReport` building blocks as the peer samples. Each model has a three-bar comparison: legacy before v1, v1, and unchanged legacy after v1. The table shows HTTP status, readiness attempts, final-request latency and prompt/completion/total tokens. Missing measurements stay empty.

The unknown-model HTTP 500 is an **expected rejection**, not a failed migration, and is kept out of the successful-contract latency charts. These few probes prove coexistence, not performance or throughput. The first successful request per model is a warm-up; only the second successful legacy request appears as **Legacy Before**. Warm-ups, readiness waits and earlier attempts are excluded from timings and tokens. The report saves sanitized evidence only, without subscription keys, prompts or completion content. No telemetry ingestion wait or new workbook is needed.

In [ ]:
from pathlib import Path

from apimtesting import ApimTesting

if 'legacy_after' not in locals() or 'v1_rejection' not in locals():
    raise SystemExit('Run both migration stages and the final legacy regression first.')
expected_deployments = {model['deploymentName'] for model in model_configuration}
assert all(set(results) == expected_deployments for results in (legacy_before, v1_success, legacy_after)), 'Run all probes for both pools.'
assert v1_rejection is not None, 'Run the unknown-model rejection probe.'

tests = ApimTesting('Azure OpenAI v1 Migration Tests', sample_folder, nb_helper.deployment)
probe_evidence = []
for stage, results in [('Legacy Before', legacy_before), ('v1', v1_success), ('Legacy After', legacy_after)]:
    for model_config in model_configuration:
        result = results[model_config['deploymentName']]
        label = f'{stage}: {model_config["name"]}'
        tests.verify(result.status_code, 200, f'{label} HTTP status')
        tests.verify(result.has_completion, True, f'{label} completion')
        tests.verify(result.backend_pool, model_config['poolName'], f'{label} backend pool')
        tests.verify(result.model, f'{model_config["name"]}-{model_config["version"]}', f'{label} model snapshot')
        probe_evidence.append(migration_helpers.ProbeEvidence(stage, model_config['name'], result))
tests.verify(v1_rejection.status_code, 500, 'Unknown model: gateway rejection')
tests.verify(v1_rejection.has_completion, False, 'Unknown model: no fallback completion')
probe_evidence.append(migration_helpers.ProbeEvidence('v1 Rejection', 'Unknown model', v1_rejection, expected_status = 500))
tests.print_summary()
assert tests.tests_failed == 0, 'Migration checks failed; rerun the affected probe before reporting success.'

report_path = migration_helpers.generate_report(
    probe_evidence, Path(utils.get_project_root()) / 'tmp' / f'{sample_name}-report.html',
)
print_ok(f'Local migration report: {report_path.as_uri()}')

### 🧹 Clean Up and Next Steps

The notebook does not automatically delete persistent lab resources or shared infrastructure. Follow [RUNBOOK.md](RUNBOOK.md#clean-up) for sample-only removal, or the selected infrastructure cleanup notebook if the entire resource group is disposable. HTTP sessions close even when smoke tests fail.

Optional: inspect [migration-requests.kql](queries/migration-requests.kql) in the existing infrastructure workspace. No additional logging resources or sensitive body logging are deployed. Streaming is passed through without replay, but this notebook only validates non-streaming chat completions.